# 🚀 GitTrend — Spark Analysis
## Topik 7: Monitor Repositori Open Source Populer
**Pipeline: HDFS → PySpark (DataFrame API + Spark SQL) → HDFS + Dashboard JSON**

### 3 Analisis Wajib:
1. **Distribusi bahasa pemrograman** — bahasa apa yang paling banyak digunakan?
2. **Top 10 repositori berdasarkan bintang** — repo mana yang paling populer?
3. **Kata trending di deskripsi repo** — tema apa yang sedang tren?

## 0. Setup & Instalasi
Install PySpark di Google Colab dan upload data dari pipeline Kafka → HDFS.

In [ ]:
!pip install pyspark -q
print('PySpark installed ✅')

In [ ]:
import os
import json
import re
from datetime import datetime
from google.colab import files

# Buat direktori staging
os.makedirs('/content/data/api', exist_ok=True)
os.makedirs('/content/data/rss', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

print('📂 Upload file JSON dari folder tmp/spark_staging/api/')
print('   (Select semua file .json dari folder api, lalu klik Open)')
uploaded_api = files.upload()
for fname, content in uploaded_api.items():
    with open(f'/content/data/api/{fname}', 'wb') as f:
        f.write(content)
print(f'\n✅ {len(uploaded_api)} file API uploaded')

In [ ]:
print('📂 Upload file JSON dari folder tmp/spark_staging/rss/')
print('   (Select semua file .json dari folder rss, lalu klik Open)')
uploaded_rss = files.upload()
for fname, content in uploaded_rss.items():
    with open(f'/content/data/rss/{fname}', 'wb') as f:
        f.write(content)
print(f'\n✅ {len(uploaded_rss)} file RSS uploaded')

## 1. Inisialisasi PySpark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GitTrend Analysis') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'✅ Spark version: {spark.version}')
print(f'✅ SparkSession ready')

## 2. Load & Eksplorasi Data
Data diambil dari file JSON yang sudah di-download dari HDFS (hasil pipeline Kafka Consumer).

In [ ]:
# Load API data (GitHub repositories)
api_records = []
for f in os.listdir('/content/data/api'):
    if f.endswith('.json'):
        with open(f'/content/data/api/{f}', 'r', encoding='utf-8') as fp:
            data = json.load(fp)
            if isinstance(data, list):
                api_records.extend(data)
            else:
                api_records.append(data)

# Load RSS data
rss_records = []
for f in os.listdir('/content/data/rss'):
    if f.endswith('.json'):
        with open(f'/content/data/rss/{f}', 'r', encoding='utf-8') as fp:
            data = json.load(fp)
            if isinstance(data, list):
                rss_records.extend(data)
            else:
                rss_records.append(data)

print(f'Raw API records: {len(api_records)}')
print(f'Raw RSS records: {len(rss_records)}')

In [ ]:
# Convert ke Spark DataFrame
df_api = spark.createDataFrame(api_records)
df_rss = spark.createDataFrame(rss_records) if rss_records else None

print(f'\n📊 API DataFrame: {df_api.count()} rows')
print('Schema:')
df_api.printSchema()

print('\n📰 Sample API data:')
df_api.select('full_name', 'language', 'stargazers_count', 'description').show(5, truncate=50)

In [ ]:
if df_rss:
    print(f'📰 RSS DataFrame: {df_rss.count()} rows')
    df_rss.printSchema()
    df_rss.show(5, truncate=50)
else:
    print('⚠️ No RSS data available')

## 3. Register Spark SQL Tables
Membuat temporary view untuk query SQL.

In [ ]:
df_api.createOrReplaceTempView('github_repos')
if df_rss:
    df_rss.createOrReplaceTempView('tech_news')

# Quick stats via SQL
print('📈 Quick Statistics:')
spark.sql("""
    SELECT 
        COUNT(*) as total_repos,
        COUNT(DISTINCT language) as unique_languages,
        ROUND(AVG(stargazers_count), 1) as avg_stars,
        MAX(stargazers_count) as max_stars,
        ROUND(AVG(forks_count), 1) as avg_forks
    FROM github_repos
""").show()

---
## 📊 Analisis 1: Distribusi Bahasa Pemrograman
**Metode:** Spark SQL `GROUP BY` + `ORDER BY`

**Pertanyaan:** Bahasa pemrograman apa yang paling banyak digunakan di repositori trending GitHub?

In [ ]:
# Analisis 1: Distribusi bahasa pemrograman (Spark SQL)
df_lang = spark.sql("""
    SELECT 
        language,
        COUNT(*) as repo_count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM github_repos), 1) as percentage,
        ROUND(AVG(stargazers_count), 1) as avg_stars,
        SUM(forks_count) as total_forks
    FROM github_repos
    GROUP BY language
    ORDER BY repo_count DESC
""")

print('🔤 Distribusi Bahasa Pemrograman di Repositori Trending:')
print('=' * 70)
df_lang.show(20, truncate=False)

# Simpan hasil ke list untuk JSON export
lang_results = [row.asDict() for row in df_lang.collect()]

### 📝 Interpretasi Analisis 1

Distribusi bahasa pemrograman menunjukkan tren teknologi yang sedang berkembang di ekosistem open source.
- **JavaScript/TypeScript** mendominasi, menunjukkan dominasi ekosistem web dan full-stack development.
- **Python** tetap populer berkat AI/ML dan automation.
- Kehadiran bahasa seperti **C++** dan **C#** menunjukkan diversitas use-case dari game modding hingga desktop tooling.
- Kategori **Unknown** menandakan banyak repository yang bersifat dokumentasi atau kumpulan resource tanpa kode.

---
## ⭐ Analisis 2: Top 10 Repositori Berdasarkan Bintang
**Metode:** DataFrame API — `orderBy()` + `limit()`

**Pertanyaan:** Repositori mana yang paling populer berdasarkan jumlah stars?

In [ ]:
# Analisis 2: Top 10 repo berdasarkan stars (DataFrame API)
df_top10 = df_api \
    .select('full_name', 'language', 'stargazers_count', 'forks_count', 'description') \
    .orderBy(F.col('stargazers_count').desc()) \
    .limit(10)

# Tambah rank menggunakan Window Function
window_spec = Window.orderBy(F.col('stargazers_count').desc())
df_top10_ranked = df_top10.withColumn('rank', F.row_number().over(window_spec))

print('⭐ Top 10 Repositori Paling Populer (by Stars):')
print('=' * 80)
df_top10_ranked.select('rank', 'full_name', 'language', 'stargazers_count', 'forks_count') \
    .show(10, truncate=False)

# Detail dengan deskripsi
print('\n📋 Detail Top 10 dengan Deskripsi:')
for row in df_top10_ranked.collect():
    desc = (row['description'] or 'No description')[:80]
    print(f"  #{row['rank']} ⭐{row['stargazers_count']} | {row['full_name']}")
    print(f"     Language: {row['language']} | Forks: {row['forks_count']}")
    print(f"     {desc}")
    print()

# Simpan hasil
top10_results = [row.asDict() for row in df_top10_ranked.collect()]

### 📝 Interpretasi Analisis 2

Top 10 repositori trending menunjukkan pola menarik:
- Repository dengan **star tertinggi** biasanya terkait tools AI, automation, atau meme/viral content.
- **Rasio forks/stars** rendah menunjukkan ini adalah proyek baru yang mendapat perhatian cepat.
- Banyak repo trending merupakan **AI-powered tools** (Claude Code skills, LLM inference, trading bots).
- Beberapa repo viral karena **gaming/modding community** (FiveM, Subnautica, PS5 exploits).

---
## 📝 Analisis 3: Kata Trending di Deskripsi Repo
**Metode:** DataFrame API — `split()` + `explode()` + `filter()` + `groupBy()`

**Pertanyaan:** Tema atau kata kunci apa yang sedang tren berdasarkan deskripsi repositori?

In [ ]:
# Analisis 3: Kata trending di deskripsi (DataFrame API)

# Stop words yang akan difilter
stop_words = [
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'is', 'it', 'that', 'this', 'are', 'was',
    'be', 'has', 'have', 'had', 'not', 'no', 'can', 'will', 'do', 'if',
    'your', 'you', 'we', 'they', 'all', 'any', 'as', 'up', 'out', 'so',
    'its', 'than', 'then', 'into', 'over', 'also', 'just', 'more', 'about',
    'one', 'two', 'new', 'use', 'using', 'used', 'get', 'set', 'via', 'etc',
    '', '-', '--', '—', '|', '→', 'https', 'http', 'www', 'com'
]

# Pipeline: filter null → lowercase → split kata → explode → filter stop words → count
df_words = df_api \
    .filter(F.col('description').isNotNull()) \
    .filter(F.col('description') != '') \
    .select(F.explode(
        F.split(
            F.lower(F.regexp_replace(F.col('description'), r'[^a-zA-Z\s]', '')),
            r'\s+'
        )
    ).alias('word')) \
    .filter(~F.col('word').isin(stop_words)) \
    .filter(F.length('word') >= 3) \
    .groupBy('word') \
    .agg(F.count('*').alias('frequency')) \
    .orderBy(F.col('frequency').desc())

print('🔥 Top 25 Kata Trending di Deskripsi Repositori:')
print('=' * 50)
df_words.show(25, truncate=False)

# Simpan hasil
word_results = [row.asDict() for row in df_words.limit(30).collect()]

### 📝 Interpretasi Analisis 3

Word frequency analysis mengungkap tema-tema yang sedang tren:
- **"AI", "code", "skill"** mendominasi — menunjukkan era AI-assisted development.
- **"bot", "trading", "market"** muncul karena tren crypto/prediction market bots.
- **"video", "claude", "codex"** mencerminkan adopsi tools AI generatif (Claude, OpenAI Codex).
- **"desktop", "app", "windows"** menunjukkan minat pada aplikasi desktop native.

**Insight utama:** Ekosistem open source saat ini didominasi oleh AI tools dan automation — developer tidak hanya *menggunakan* AI, tapi aktif *membangun* tools AI.

---
## 4. Export Hasil ke JSON (untuk Dashboard)
Simpan hasil analisis dalam format JSON untuk dikonsumsi oleh Flask Dashboard (Anggota 5).

In [ ]:
# Compile semua hasil analisis
spark_results = {
    'metadata': {
        'generated_at': datetime.now().isoformat(),
        'spark_version': spark.version,
        'total_api_records': df_api.count(),
        'total_rss_records': df_rss.count() if df_rss else 0,
        'analysis_count': 3
    },
    'analysis_1_language_distribution': lang_results,
    'analysis_2_top_repos': top10_results,
    'analysis_3_trending_words': word_results
}

# Simpan ke file
output_path = '/content/output/spark_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(spark_results, f, indent=2, ensure_ascii=False, default=str)

print(f'✅ Hasil disimpan ke: {output_path}')
print(f'   File size: {os.path.getsize(output_path)} bytes')
print(f'\n📋 Ringkasan:')
print(f'   - Analisis 1: {len(lang_results)} bahasa pemrograman')
print(f'   - Analisis 2: {len(top10_results)} top repos')
print(f'   - Analisis 3: {len(word_results)} trending words')

In [ ]:
# Download spark_results.json ke komputer lokal
# Setelah download, pindahkan file ini ke:
#   kelompok-3-ets-bigdata/dashboard/data/spark_results.json

print('📥 Downloading spark_results.json...')
print('   Setelah download, letakkan di: dashboard/data/spark_results.json')
files.download('/content/output/spark_results.json')

## 5. Upload Hasil ke HDFS (via Docker)
Setelah download `spark_results.json`, jalankan command berikut di terminal lokal untuk upload ke HDFS:

```bash
# Copy file ke namenode container
docker cp dashboard/data/spark_results.json namenode:/tmp/spark_results.json

# Upload ke HDFS
docker exec namenode hdfs dfs -put -f /tmp/spark_results.json /data/github/hasil/

# Verifikasi
docker exec namenode hdfs dfs -ls /data/github/hasil/
```

In [ ]:
# Cleanup
spark.stop()
print('\n🎉 Analisis selesai!')
print('\nNext steps:')
print('1. Download spark_results.json (cell di atas)')
print('2. Pindahkan ke dashboard/data/spark_results.json')
print('3. Upload ke HDFS (command di atas)')
print('4. Lanjut ke Anggota 5: Flask Dashboard')